# 3.8: Train mode vs. evaluation mode

Two layers in this chapter behave differently in training and evaluation:

| Layer | Training mode | Evaluation mode |
|---|---|---|
| Dropout | Randomly zeros activations and rescales by $`1/(1-p)`$ | Identity (no dropout) |
| BatchNorm | Normalizes with the current batch's statistics; updates running statistics | Normalizes with the stored running statistics; no updates |
| LayerNorm, RMSNorm | Same computation | Same computation |

PyTorch tracks which mode each module is in with a flag set by `model.train()` and `model.eval()`. These calls do not start training or evaluation; they only set the flag, which propagates to every submodule.

Earlier imports and definitions this listing relies on are included below so the notebook runs on its own.

This snippet uses `train_loader`, `val_loader` from the surrounding training code. Those names are not defined in an earlier listing, so the cell shows the pattern as it appears in the chapter.


In [ ]:
# Imports and definitions from earlier listings, included so this notebook runs alone.

# From Warmup plus decay for LLM pretraining
import math

def lr_at(step, peak_lr, warmup_steps, total_steps, min_lr_ratio=0.1):
    min_lr = peak_lr * min_lr_ratio
    if step < warmup_steps:
        return peak_lr * (step + 1) / warmup_steps
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    progress = min(progress, 1.0)
    return min_lr + 0.5 * (peak_lr - min_lr) * (1 + math.cos(math.pi * progress))

# PyTorch: apply via LambdaLR, which multiplies the optimizer's base lr
import torch

model = torch.nn.Linear(10, 10)

opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.1)

sched = torch.optim.lr_scheduler.LambdaLR(
    opt, lambda step: lr_at(step, 1.0, warmup_steps=1000, total_steps=100_000))


In [ ]:
model.train()                  # dropout active, BatchNorm uses batch stats
for x, y in train_loader:
    ...

model.eval()                   # dropout off, BatchNorm uses running stats
with torch.no_grad():          # separately: don't build the autograd graph
    for x, y in val_loader:
        ...
model.train()                  # remember to switch back
